In [20]:
import json
import glob
import pandas as pd

import sys
sys.path.append("..")
from src.extract import STATE_COLUMNS

files = sorted(glob.glob("../data/raw/opensky/*.json"))
with open(files[-1], encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data["states"], columns=STATE_COLUMNS)
df["snapshot_time"] = pd.to_datetime(data["time"], unit="s", utc=True)
df.shape

(607, 18)

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 607 entries, 0 to 606
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype             
---  ------           --------------  -----             
 0   icao24           607 non-null    str               
 1   callsign         607 non-null    str               
 2   origin_country   607 non-null    str               
 3   time_position    607 non-null    int64             
 4   last_contact     607 non-null    int64             
 5   longitude        607 non-null    float64           
 6   latitude         607 non-null    float64           
 7   baro_altitude    537 non-null    float64           
 8   on_ground        607 non-null    bool              
 9   velocity         607 non-null    float64           
 10  true_track       607 non-null    float64           
 11  vertical_rate    535 non-null    float64           
 12  sensors          0 non-null      object            
 13  geo_altitude     534 non-null    float64      

In [22]:
df.isna().sum()

icao24               0
callsign             0
origin_country       0
time_position        0
last_contact         0
longitude            0
latitude             0
baro_altitude       70
on_ground            0
velocity             0
true_track           0
vertical_rate       72
sensors            607
geo_altitude        73
squawk             106
spi                  0
position_source      0
snapshot_time        0
dtype: int64

In [23]:
df["on_ground"].value_counts()

on_ground
False    537
True      70
Name: count, dtype: int64

In [24]:
df["callsign"].str.len().value_counts()

callsign
8    605
0      2
Name: count, dtype: int64

In [25]:
df["icao24"].duplicated().sum()

np.int64(0)

In [26]:
df[["velocity", "baro_altitude"]].describe()

,velocity,baro_altitude
count,607.000000,537.000000
mean,169.337430,8008.818659
std,84.851817,4087.964377
min,0.000000,-60.960000
25%,99.180000,4572.000000
50%,209.790000,10363.200000
75%,232.120000,11269.980000
max,282.410000,13716.000000


In [27]:
df["origin_country"].value_counts().head(10)

origin_country
Spain             118
France             71
Ireland            62
Malta              48
United Kingdom     40
Portugal           38
Germany            34
Morocco            28
Austria            25
United States      15
Name: count, dtype: int64

## Constats (snapshot de 607 avions)
- Aucune latitude/longitude manquante (la bounding box filtre déjà les avions sans position)
- `sensors` est vide à 100 % → à supprimer
- Valeurs nulles : baro_altitude (70), vertical_rate (72), geo_altitude (73), squawk (106)
- 537 avions en vol, 70 au sol
- `callsign` : 605 valeurs de 8 caractères (complétées par des espaces) et 2 vides → strip() nécessaire
- Aucun doublon d'icao24 dans ce snapshot
- Vitesse (m/s) : 0 à 282 ; altitude (m) : -61 à 13 716, valeurs plausibles
- `origin_country` = pays d'enregistrement de l'avion, pas pays de départ

- airlines.dat : 275 compagnies sans code ICAO (dont les "N/A" convertis en NaN par pandas)
- airlines.dat : 4 906 inactives, 1 255 actives, 1 valeur "n" en minuscule à normaliser

In [28]:
df[df["on_ground"]]["baro_altitude"].isna().sum()

np.int64(70)

In [29]:
from src.extract import extract_airlines
airlines = extract_airlines("../data/raw/airlines.dat")
airlines["icao"].isna().sum(), airlines["icao"].isin(["N/A", "", "-"]).sum()

2026-10-12 01:00:57,869 | INFO | extract | Airlines extracted: 6162 rows


(np.int64(275), np.int64(0))

In [30]:
airlines["active"].value_counts()

active
N    4906
Y    1255
n       1
Name: count, dtype: int64

In [31]:
import pandas as pd
from src.utils import get_logger

logger = get_logger("transform")


def clean_states(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    n_start = len(df)

    df = df.drop(columns=["sensors"])
    df["callsign"] = df["callsign"].str.strip()
    df["callsign"] = df["callsign"].mask(df["callsign"] == "")
    df = df.dropna(subset=["icao24", "latitude", "longitude"])
    df = df.drop_duplicates(subset=["icao24", "time_position"])
    df = df.rename(columns={"origin_country": "registration_country"})

    logger.info("clean_states: %d -> %d rows", n_start, len(df))
    return df.reset_index(drop=True)


def convert_units(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["speed_kmh"] = df["velocity"] * 3.6
    df["altitude_ft"] = df["baro_altitude"] * 3.28084
    df["time_position"] = pd.to_datetime(df["time_position"], unit="s", utc=True)
    df["last_contact"] = pd.to_datetime(df["last_contact"], unit="s", utc=True)
    return df.drop(columns=["velocity", "baro_altitude"])


def extract_airline_code(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # 3 lettres suivies d'un chiffre : TVF73DJ -> TVF
    df["airline_icao"] = df["callsign"].str.extract(r"^([A-Z]{3})\d", expand=False)
    return df


def clean_airlines(airlines: pd.DataFrame) -> pd.DataFrame:
    df = airlines.copy()
    df["active"] = df["active"].str.upper()
    df = df.dropna(subset=["icao"])
    df["icao"] = df["icao"].str.upper().str.strip()
    # Garder la version active en premier quand un ICAO est dupliqué
    df = df.sort_values("active", ascending=False)   # "Y" avant "N"
    df = df.drop_duplicates(subset="icao", keep="first")
    df = df.rename(columns={"name": "airline_name", "country": "airline_country"})
    logger.info("clean_airlines: %d -> %d rows", len(airlines), len(df))
    return df[["icao", "airline_name", "iata", "airline_country", "active"]]


def join_airlines(states: pd.DataFrame, airlines_clean: pd.DataFrame) -> pd.DataFrame:
    n_before = len(states)
    out = states.merge(
        airlines_clean,
        how="left",
        left_on="airline_icao",
        right_on="icao",
    ).drop(columns=["icao"])
    assert len(out) == n_before, "La jointure a dupliqué des lignes"
    return out

In [32]:
%load_ext autoreload
%autoreload 2

from src.transform import (
    clean_states, convert_units, extract_airline_code,
    clean_airlines, join_airlines,
)

states = extract_airline_code(convert_units(clean_states(df)))
airlines_clean = clean_airlines(airlines)
joined = join_airlines(states, airlines_clean)

print(joined.shape)
joined[["callsign", "airline_icao", "airline_name", "airline_country"]].head(10)

2026-10-12 01:00:58,352 | INFO | transform | clean_states: 607 -> 607 rows
2026-10-12 01:00:58,395 | INFO | transform | clean_airlines: 6162 -> 5853 rows


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
(607, 22)


,callsign,airline_icao,airline_name,airline_country
0,TVF73DJ,TVF,Transavia France,France
1,TVF28LP,TVF,Transavia France,France
2,TVF43EV,TVF,Transavia France,France
3,TVF89CZ,TVF,Transavia France,France
4,EJU1713,EJU,NaN,NaN
5,FCVRB,NaN,NaN,NaN
6,NSZ9PR,NSZ,NaN,NaN
7,ANE007,ANE,Air Nostrum,Spain
8,MBU3WL,MBU,NaN,NaN
9,ANE79GL,ANE,Air Nostrum,Spain


In [33]:
print("Sans code compagnie :", joined["airline_icao"].isna().sum())
print("Code mais compagnie inconnue :", (joined["airline_icao"].notna() & joined["airline_name"].isna()).sum())
joined["airline_name"].value_counts().head(10)

Sans code compagnie : 62
Code mais compagnie inconnue : 82


airline_name
Ryanair             88
Vueling Airlines    40
TAP Portugal        30
Royal Air Maroc     24
Iberia Airlines     18
Transavia France    17
Air Nostrum         16
Eurowings           15
Jet2.com            13
easyJet             13
Name: count, dtype: int64

In [34]:
%load_ext autoreload
%autoreload 2

from src.transform import (
    clean_states, convert_units, extract_airline_code,
    clean_airlines, join_airlines,
)

states = extract_airline_code(convert_units(clean_states(df)))
airlines_clean = clean_airlines(airlines)
joined = join_airlines(states, airlines_clean)

print(joined.shape)
joined[["callsign", "airline_icao", "airline_name", "airline_country"]].head(10)

2026-10-12 01:00:58,742 | INFO | transform | clean_states: 607 -> 607 rows
2026-10-12 01:00:58,778 | INFO | transform | clean_airlines: 6162 -> 5853 rows


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
(607, 22)


,callsign,airline_icao,airline_name,airline_country
0,TVF73DJ,TVF,Transavia France,France
1,TVF28LP,TVF,Transavia France,France
2,TVF43EV,TVF,Transavia France,France
3,TVF89CZ,TVF,Transavia France,France
4,EJU1713,EJU,NaN,NaN
5,FCVRB,NaN,NaN,NaN
6,NSZ9PR,NSZ,NaN,NaN
7,ANE007,ANE,Air Nostrum,Spain
8,MBU3WL,MBU,NaN,NaN
9,ANE79GL,ANE,Air Nostrum,Spain


In [35]:
%autoreload 2
from src.transform import build_star_schema

tables = build_star_schema(joined)
for name, t in tables.items():
    print(name, t.shape)

tables["dim_airline"].head(10)

2026-10-12 01:00:58,970 | INFO | transform | dim_time: 1 rows
2026-10-12 01:00:58,973 | INFO | transform | dim_country: 45 rows
2026-10-12 01:00:58,975 | INFO | transform | dim_aircraft: 607 rows
2026-10-12 01:00:58,976 | INFO | transform | dim_airline: 114 rows
2026-10-12 01:00:58,977 | INFO | transform | fact_flight_states: 607 rows


dim_time (1, 8)
dim_country (45, 1)
dim_aircraft (607, 2)
dim_airline (114, 4)
fact_flight_states (607, 16)


,airline_icao,airline_name,iata,airline_country
0,TVF,Transavia France,TO,France
1,EJU,Unknown,NaN,NaN
2,NSZ,Unknown,NaN,NaN
3,ANE,Air Nostrum,YW,Spain
4,MBU,Unknown,NaN,NaN
5,EWG,Eurowings,EW,Germany
6,VLG,Vueling Airlines,VY,Spain
7,IBS,Iberia Express,I2,Spain
8,AEA,Air Europa,UX,Spain
9,RVP,Air VIP,NaN,Portugal


In [36]:
from src.load import ROOT

p = ROOT / "sql"
print(ROOT)
print(p.exists(), [f.name for f in p.glob("*")])

C:\Users\farah\Documents\etl-flights-pipeline
True []


In [19]:
%autoreload 2
from src.transform import build_star_schema

tables = build_star_schema(joined)
print(tables["dim_country"].columns.tolist())
print(tables["dim_aircraft"].columns.tolist())

2026-10-12 01:00:55,746 | INFO | transform | dim_time: 1 rows
2026-10-12 01:00:55,751 | INFO | transform | dim_country: 45 rows
2026-10-12 01:00:55,752 | INFO | transform | dim_aircraft: 607 rows
2026-10-12 01:00:55,753 | INFO | transform | dim_airline: 114 rows
2026-10-12 01:00:55,755 | INFO | transform | fact_flight_states: 607 rows


['country_name']
['icao24', 'registration_country']


In [42]:
from src.load import ROOT

print(ROOT)
sql_dir = ROOT / "sql"
print("sql/ existe ?", sql_dir.exists())
print("Contenu de sql/ :", [f.name for f in sql_dir.glob("*")] if sql_dir.exists() else None)

for p in ROOT.rglob("create_tables*"):
    if "venv" not in p.parts:
        print("Trouvé :", p)

C:\Users\farah\Documents\etl-flights-pipeline
sql/ existe ? True
Contenu de sql/ : []
Trouvé : C:\Users\farah\Documents\etl-flights-pipeline\src\create_tables.sql


In [43]:
import shutil
from src.load import ROOT

src_file = ROOT / "src" / "create_tables.sql"
dst_file = ROOT / "sql" / "create_tables.sql"

shutil.move(str(src_file), str(dst_file))

print("sql/ contient :", [f.name for f in (ROOT / "sql").glob("*")])
print("src/ contient create_tables ?", (ROOT / "src" / "create_tables.sql").exists())

sql/ contient : ['create_tables.sql']
src/ contient create_tables ? False


In [46]:
%autoreload 2
from src.load import get_engine, create_tables, load_staging, load_warehouse

engine = get_engine()
create_tables(engine)
load_staging(engine, df)
load_warehouse(engine, tables)

2026-10-12 01:23:34,399 | INFO | load | Schemas and tables ready
2026-10-12 01:23:34,543 | INFO | load | staging.opensky_states: 607 rows loaded (snapshot 2026-10-05 13:07:15+00:00)
2026-10-12 01:23:34,652 | INFO | load | dim_country: 45 rows sent, 0 new (total 45)
2026-10-12 01:23:34,692 | INFO | load | dim_time: 1 rows sent, 0 new (total 1)
2026-10-12 01:23:34,747 | INFO | load | dim_airline: 114 rows sent, 0 new (total 114)
2026-10-12 01:23:34,912 | INFO | load | dim_aircraft: 607 rows sent, 0 new (total 607)
2026-10-12 01:23:35,307 | INFO | load | fact_flight_states: 607 rows sent, 0 new (total 607)
